## for testing code and planning

---
###	Test	Description
- 1	Setup	Configure URLs and test environment
- 2	API Health	Check all services (Gateway, OCR, RAG, Classifier, Verification)
---

- 3	OCR Engines	Check EasyOCR/PaddleOCR availability and config
- 4	OCR Extraction	Test text file extraction with entity detection
---

- 5	RAG Health	Check RAG service and ChromaDB status
- 6	RAG Retrieval	Test 3 queries (fraud, harassment, hacking) with simple output
---

- 7	Summary View	Visual dashboard of all test results
- 8	Full Pipeline	Complete analysis example with evidence file

### requests api

In [8]:
import requests
import json
from pprint import pprint

API_URL = "https://cyber-crime-production.up.railway.app/"
OCR_URL = "http://localhost:8001"
RAG_URL = "http://localhost:8003"

TEST_EMAIL = "test@example.com"
TEST_PASSWORD = "testpassword123"

print("Test environment configured")
print(f"   API Gateway: {API_URL}")
print(f"   OCR Service: {OCR_URL}")
print(f"   RAG Service: {RAG_URL}")

Test environment configured
   API Gateway: https://cyber-crime-production.up.railway.app/
   OCR Service: http://localhost:8001
   RAG Service: http://localhost:8003


In [2]:
def test_api_health():
    """Test API Gateway and all downstream services"""
    print("Testing API Gateway Health...")
    
    try:
        response = requests.get(f"{API_URL}/health", timeout=10)
        data = response.json()
        
        print(f"\nGateway Status: {data.get('gateway', 'unknown')}")
        print(f"Database: {data.get('database', 'unknown')}")
        
        print("\nServices Status:")
        services = data.get('services', {})
        for service, status in services.items():
            icon = "true" if status == "healthy" else "false"
            print(f"   {icon} {service}: {status}")
        
        # Check if critical services are up
        critical = ['ocr', 'classifier', 'rag', 'verification']
        all_healthy = all(services.get(s) == "healthy" for s in critical)
        
        if all_healthy:
            print("\nAll critical services are healthy!")
        else:
            print("\n Some services are not healthy")
        
        return data
        
    except Exception as e:
        print(f"\nAPI Health Check Failed: {e}")
        return None

# Run the test
api_health = test_api_health()

Testing API Gateway Health...

API Health Check Failed: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /health (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8000): Failed to establish a new connection: [Errno 111] Connection refused"))


### OCR service

In [3]:
def test_ocr_service():
    """Test OCR service with a sample image"""
    print("Testing OCR Service...")
    try:
        response = requests.get(f"{OCR_URL}/engines/status", timeout=5)
        status = response.json()
        
        print("\n OCR Engines Status:")
        print(f"   Initialized: {status.get('initialized', False)}")
        print(f"   EasyOCR: {status.get('easyocr', {}).get('available', False)}")
        print(f"   PaddleOCR: {status.get('paddleocr', {}).get('available', False)}")
        
        if status.get('config'):
            cfg = status['config']
            print(f"\nConfig:")
            print(f"   Confidence Threshold: {cfg.get('confidence_threshold')}")
            print(f"   Preprocessing: {cfg.get('use_preprocessing')}")
            print(f"   Target Width: {cfg.get('target_width')}")
        
        return status
        
    except Exception as e:
        print(f"\n OCR Status Check Failed: {e}")
        return None

# Run OCR engine status test
ocr_status = test_ocr_service()

Testing OCR Service...

 OCR Status Check Failed: HTTPConnectionPool(host='localhost', port=8001): Max retries exceeded with url: /engines/status (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8001): Failed to establish a new connection: [Errno 111] Connection refused"))


In [ ]:

def test_ocr_extraction():
    """Test OCR text extraction with a simple text file"""
    print("Testing OCR Text Extraction...")
    
    # Create a simple text file for testing
    test_text = """
    This is a test document for OCR processing.
    Phone: 01012345678
    Amount: 5000 EGP
    Date: 2024-01-15
    IBAN: EG9500190001000123456789012
    
    مشروع تجريبي باللغة العربية
    """
    
    try:
        # Create temporary text file
        import tempfile
        with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as f:
            f.write(test_text)
            temp_path = f.name
        
        # Send to OCR service through API Gateway
        with open(temp_path, 'rb') as f:
            files = {'file': ('test_evidence.txt', f, 'text/plain')}
            response = requests.post(
                f"{API_URL}/ocr/extract",
                files=files,
                timeout=30
            )
        
        # Clean up temp file
        import os
        os.unlink(temp_path)
        
        if response.status_code == 200:
            result = response.json()
            
            print("\nOCR Result:")
            print(f"   Text Length: {len(result.get('full_text', ''))} chars")
            print(f"   Confidence: {result.get('avg_confidence', 0):.2%}")
            print(f"   Language: {result.get('language', 'unknown')}")
            print(f"   Blocks: {len(result.get('evidence_blocks', []))}")
            
            # Show metadata
            meta = result.get('processing_metadata', {})
            print(f"\nProcessing:")
            print(f"   Engine: {meta.get('engine_used', 'unknown')}")
            print(f"   Fallback: {meta.get('fallback_triggered', False)}")
            print(f"   Time: {meta.get('processing_time_ms', 0):.0f}ms")
            
            # Show entities
            entities = result.get('entities', {})
            if entities:
                print(f"\nExtracted Entities:")
                for key, items in entities.items():
                    if items:
                        print(f"   {key}: {len(items)} found")
                        for item in items[:3]:  # Show first 3
                            val = item.get('value', item) if isinstance(item, dict) else item
                            print(f"      - {val}")
            
            # Show confidence score details
            conf_score = meta.get('confidence_score')
            if conf_score:
                print(f"\nConfidence Score:")
                print(f"   Status: {conf_score.get('status', 'unknown')}")
                print(f"   Weighted Avg: {conf_score.get('weighted_average', 0):.2%}")
                print(f"   Minimum: {conf_score.get('minimum', 0):.2%}")
            
            return result
        else:
            print(f"\nOCR Request Failed: {response.status_code}")
            print(f"   {response.text[:200]}")
            return None
            
    except Exception as e:
        print(f"\nOCR Extraction Test Failed: {e}")
        return None

# Run OCR extraction test
ocr_result = test_ocr_extraction()

### RAG service

In [4]:
def test_rag_service():
    """Test RAG service retrieval"""
    print("Testing RAG Service...")
    
    try:
        # Check RAG health
        response = requests.get(f"{RAG_URL}/health", timeout=5)
        health = response.json()
        
        print(f"\nRAG Status: {health.get('status', 'unknown')}")
        print(f"   Version: {health.get('version', 'unknown')}")
        print(f"   Vector DB: {health.get('vector_db', 'unknown')}")
        
        # Check if Chroma is connected
        chroma = health.get('chroma', {})
        if 'error' in chroma:
            print(f"   Chroma: {chroma['error']}")
        else:
            print(f"   Chroma: Connected")
        
        return health
        
    except Exception as e:
        print(f"\nRAG Health Check Failed: {e}")
        return None

# Run RAG health test
rag_health = test_rag_service()

Testing RAG Service...

RAG Health Check Failed: HTTPConnectionPool(host='localhost', port=8003): Max retries exceeded with url: /health (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8003): Failed to establish a new connection: [Errno 111] Connection refused"))


In [5]:

def test_rag_retrieval():
    """Test RAG article retrieval with a cybercrime query"""
    print("Testing RAG Article Retrieval...")
    
    # Test queries for different crime types
    test_queries = [
        {
            "query": "الاحتيال الإلكتروني وسرقة البيانات الشخصية",
            "crime_type": "fraud",
            "description": "Electronic fraud and data theft"
        },
        {
            "query": "تهديدات عبر وسائل التواصل الاجتماعي",
            "crime_type": "harassment",
            "description": "Social media threats"
        },
        {
            "query": "قرصنة الحسابات البنكية",
            "crime_type": "hacking",
            "description": "Bank account hacking"
        }
    ]
    
    results = []
    
    for test in test_queries:
        print(f"\nQuery: {test['description']}")
        print(f"   Text: {test['query'][:50]}...")
        
        try:
            payload = {
                "query": test['query'],
                "crime_type": test['crime_type'],
                "top_k": 3,
                "tenant_id": "default",
                "transform_strategy": "auto"
            }
            
            response = requests.post(
                f"{API_URL}/retrieve",
                json=payload,
                timeout=30
            )
            
            if response.status_code == 200:
                result = response.json()
                articles = result.get('articles', [])
                
                print(f" Retrieved {len(articles)} articles")
                print(f"   Cache: {result.get('cache_hit', False)}")
                print(f"   Strategy: {result.get('query_strategy', 'none')}")
                print(f"   Latency: {result.get('latency_ms', 0):.0f}ms")
                
                # Show top article
                if articles:
                    top = articles[0]
                    print(f"\n  Top Match:")
                    print(f"      Article: {top.get('article_number', 'N/A')}")
                    print(f"      Law: {top.get('law', 'N/A')}")
                    print(f"      Relevance: {(1 - top.get('relevance_score', 1)):.1%}")
                    text = top.get('text', '')[:100]
                    print(f"      Preview: {text}...")
                    if top.get('penalty_ar'):
                        print(f"      Penalty: {top.get('penalty_ar')[:80]}...")
                
                results.append({
                    'query': test['description'],
                    'articles_found': len(articles),
                    'cache_hit': result.get('cache_hit', False),
                    'latency_ms': result.get('latency_ms', 0)
                })
            else:
                print(f"   Request Failed: {response.status_code}")
                
        except Exception as e:
            print(f"   Error: {e}")
    
    # Summary
    print("\n" + "="*60)
    print(" RAG RETRIEVAL SUMMARY")
    print("="*60)
    total_articles = sum(r['articles_found'] for r in results)
    avg_latency = sum(r['latency_ms'] for r in results) / len(results) if results else 0
    
    print(f"Total Queries: {len(results)}")
    print(f"Total Articles: {total_articles}")
    print(f"Average Latency: {avg_latency:.0f}ms")
    
    return results

# Run RAG retrieval test
rag_results = test_rag_retrieval()

Testing RAG Article Retrieval...

Query: Electronic fraud and data theft
   Text: الاحتيال الإلكتروني وسرقة البيانات الشخصية...
   Error: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /retrieve (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8000): Failed to establish a new connection: [Errno 111] Connection refused"))

Query: Social media threats
   Text: تهديدات عبر وسائل التواصل الاجتماعي...
   Error: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /retrieve (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8000): Failed to establish a new connection: [Errno 111] Connection refused"))

Query: Bank account hacking
   Text: قرصنة الحسابات البنكية...
   Error: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /retrieve (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8000): Failed to establish a new connection: [Errno 111] Connectio

### run test summary

In [6]:
def run_all_tests_summary():
    """Display comprehensive test results"""
    
    print("╔" + "="*70 + "╗")
    print("║" + " "*20 + "CYBERCRIME AI - TEST SUMMARY" + " "*20 + "║")
    print("╚" + "="*70 + "╝")
    
    # API Health
    print("\n📡 API GATEWAY")
    print("-" * 70)
    if api_health:
        services = api_health.get('services', {})
        healthy = sum(1 for s in services.values() if s == 'healthy')
        total = len(services)
        print(f"   Services: {healthy}/{total} healthy")
        for name, status in services.items():
            icon = "🟢" if status == 'healthy' else "🔴"
            print(f"   {icon} {name:15} → {status}")
    else:
        print("   API Gateway not responding")
    
    # OCR Service
    print("\n OCR SERVICE")
    print("-" * 70)
    if ocr_status:
        print(f"   Initialized: {'🟢 Yes' if ocr_status.get('initialized') else '🔴 No'}")
        print(f"   EasyOCR: {'🟢 Available' if ocr_status.get('easyocr', {}).get('available') else '🔴 Unavailable'}")
        print(f"   PaddleOCR: {'🟢 Available' if ocr_status.get('paddleocr', {}).get('available') else '🔴 Unavailable'}")
    else:
        print("    OCR Service not responding")
    
    if ocr_result:
        print(f"\n   Last Extraction:")
        print(f"    Confidence: {ocr_result.get('avg_confidence', 0):.1%}")
        print(f"   Language: {ocr_result.get('language', 'unknown')}")
        print(f"   Blocks: {len(ocr_result.get('evidence_blocks', []))}")
        meta = ocr_result.get('processing_metadata', {})
        print(f"   Engine: {meta.get('engine_used', 'unknown')}")
        print(f"   Fallback: {'Yes' if meta.get('fallback_triggered') else 'No'}")
    
    # RAG Service
    print("\nRAG SERVICE")
    print("-" * 70)
    if rag_health:
        print(f"   Status: 🟢 {rag_health.get('status', 'unknown')}")
        print(f"   Vector DB: {rag_health.get('vector_db', 'unknown')}")
    else:
        print("   🔴 RAG Service not responding")
    
    if rag_results:
        total_articles = sum(r['articles_found'] for r in rag_results)
        avg_latency = sum(r['latency_ms'] for r in rag_results) / len(rag_results)
        print(f"\n   Retrieval Performance:")
        print(f"   Total Articles: {total_articles}")
        print(f"   Avg Latency: {avg_latency:.0f}ms")
        print(f"   Queries Tested: {len(rag_results)}")
    
    # Overall Status
    print("\n" + "="*70)
    all_ok = (
        api_health and 
        ocr_status and ocr_status.get('initialized') and
        rag_health and rag_health.get('status') == 'healthy'
    )
    
    if all_ok:
        print("ALL SYSTEMS OPERATIONAL")
    else:
        print(" SOME SERVICES NEED ATTENTION")
    print("="*70)

# Run summary
run_all_tests_summary()

╔======================================================================╗
║                    CYBERCRIME AI - TEST SUMMARY                    ║
╚======================================================================╝

📡 API GATEWAY
----------------------------------------------------------------------
   API Gateway not responding

 OCR SERVICE
----------------------------------------------------------------------
    OCR Service not responding


NameError: name 'ocr_result' is not defined

### full analysis

In [7]:


def test_full_analysis():
    """Example: Run full cybercrime analysis through API Gateway"""
    print(" Running Full Pipeline Analysis Test...")
    
    # Create test evidence as text file
    evidence_text = """
    CASE: Electronic Fraud Report
    
    Complainant: Ahmed Hassan
    Phone: 01098765432
    Date: 2024-03-15
    
    INCIDENT DETAILS:
    Received WhatsApp message from number +20 10 1234 5678
    claiming to be from bank. Asked for OTP and card details.
    Amount stolen: 15,000 EGP from account.
    IBAN: EG8500190002000123456789012
    
    EVIDENCE:
    - Screenshot of conversation attached
    - Bank statement showing transaction
    - Voice recording of follow-up call
    
    مشروع الاحتيال الإلكتروني
    """
    
    try:
        import tempfile
        import os
        
        # Create temp file
        with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as f:
            f.write(evidence_text)
            temp_path = f.name
        
        print(f"\nCreated test evidence: {len(evidence_text)} chars")
        
        # Send to API for analysis (async mode)
        with open(temp_path, 'rb') as f:
            files = {'files': ('fraud_case.txt', f, 'text/plain')}
            
            print("Sending to /analyze (async)...")
            response = requests.post(
                f"{API_URL}/analyze",
                files=files,
                timeout=30
            )
        
        os.unlink(temp_path)
        
        if response.status_code == 200:
            result = response.json()
            case_id = result.get('case_id')
            
            print(f"\nAnalysis Started!")
            print(f"   Case ID: {case_id}")
            print(f"   Status: {result.get('status')}")
            
            # For demo, try sync endpoint instead
            print("\nRunning sync analysis for immediate results...")
            
            with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as f:
                f.write(evidence_text)
                temp_path = f.name
            
            with open(temp_path, 'rb') as f:
                files = {'files': ('fraud_case.txt', f, 'text/plain')}
                response = requests.post(
                    f"{API_URL}/analyze/json",
                    files=files,
                    timeout=120
                )
            
            os.unlink(temp_path)
            
            if response.status_code == 200:
                analysis = response.json()
                
                print("\n" + "="*60)
                print(" ANALYSIS RESULTS")
                print("="*60)
                
                # Classification
                clf = analysis.get('classification', {})
                print(f"\n Crime Type: {clf.get('crime_type', 'unknown')}")
                print(f"   Confidence: {clf.get('confidence', 0):.1%}")
                
                # Entities
                entities = analysis.get('entities', {})
                print(f"\nEntities Found:")
                for key, items in entities.items():
                    if items:
                        print(f"   {key}: {len(items)}")
                        for item in items[:2]:
                            val = item.get('value', item) if isinstance(item, dict) else item
                            print(f"      • {val}")
                
                # Articles
                articles = analysis.get('articles', [])
                print(f"\nRelevant Laws: {len(articles)} articles")
                for art in articles[:3]:
                    print(f"   • Article {art.get('article_number')} ({art.get('law')})")
                
                # Score
                score = analysis.get('score', {})
                print(f"\n Case Strength: {score.get('total_score', 0)}% ({score.get('grade', 'N/A')})")
                
                # Verification
                verif = analysis.get('verification', {})
                print(f"   Verification: {verif.get('status')} ({verif.get('rounds')} rounds)")
                
                # OCR Details
                ocr = analysis.get('ocr', {})
                print(f"\n OCR: {ocr.get('avg_confidence', 0):.1%} confidence")
                
                return analysis
            else:
                print(f" Sync analysis failed: {response.status_code}")
                return None
        else:
            print(f"Analysis request failed: {response.status_code}")
            return None
            
    except Exception as e:
        print(f"\n Full Analysis Test Failed: {e}")
        import traceback
        traceback.print_exc()
        return None

# Run full analysis test (uncomment to run)
full_result = test_full_analysis()

 Running Full Pipeline Analysis Test...

Created test evidence: 509 chars
Sending to /analyze (async)...

 Full Analysis Test Failed: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /analyze (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8000): Failed to establish a new connection: [Errno 111] Connection refused"))


Traceback (most recent call last):
  File "/home/youssef/miniconda3/envs/cybercrime/lib/python3.11/site-packages/urllib3/connection.py", line 204, in _new_conn
    sock = connection.create_connection(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/youssef/miniconda3/envs/cybercrime/lib/python3.11/site-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/home/youssef/miniconda3/envs/cybercrime/lib/python3.11/site-packages/urllib3/util/connection.py", line 73, in create_connection
    sock.connect(sa)
ConnectionRefusedError: [Errno 111] Connection refused

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/home/youssef/miniconda3/envs/cybercrime/lib/python3.11/site-packages/urllib3/connectionpool.py", line 787, in urlopen
    response = self._make_request(
               ^^^^^^^^^^^^^^^^^^^
  File "/home/youssef/miniconda3/envs/cybercrime/lib/python3.11/site-packages/urllib3/conne